# Train the Phase 6 models (helmet + plate) on Kaggle

Trains the Phase 6 extension models on a Kaggle GPU (free P100 / T4 x2, 30 h/week):

- **helmet** — helmet non-use detection (`configs/train.helmet.yaml`, Roboflow helmet-detection_yolov8 v3, CC BY 4.0)
- **plate** — license-plate detector for the optional ANPR module (`configs/train.plate.yaml`, License Plate Recognition v13, CC BY 4.0)

The Phase 1 vehicle detector is already trained and committed
(`weights/detector.pt`, `docs/RESULTS.md`) — no need to re-run it here.

**Before running:**
1. Right sidebar → **Add-ons → Secrets** → add `ROBOFLOW_API_KEY`
   (key from https://app.roboflow.com/settings/api). Never paste the key in a cell.
2. Right sidebar → **Notebook options → Internet: On** (needs a phone-verified account).
3. Right sidebar → **Accelerator: GPU T4 x2** (or P100).

**Note:** the plate dataset has ~99k train images; a full 20-epoch run can approach
the 12 h session limit. If it gets cut off, re-run with fewer epochs, e.g.
`--epochs 10` appended to the plate training command.

In [ ]:
# 1. Clone the repo (Phase 6 branch) and install deps — safe to re-run this cell
import os, sys

%cd /kaggle/working
!rm -rf tvd
!git clone -b phase-6-extensions https://github.com/Rakib-codee/Traffic-Violation-Detection.git tvd
%cd /kaggle/working/tvd
!pip uninstall -y tvd 2>/dev/null || true  # drop any unrelated PyPI 'tvd' package
!pip install -q -e ".[dev,data]"

# drop 'tvd' modules cached by earlier failed imports, then make src importable
# (editable-install hooks only load at interpreter startup)
for _m in [m for m in list(sys.modules) if m == "tvd" or m.startswith("tvd.")]:
    del sys.modules[_m]
sys.path.insert(0, "/kaggle/working/tvd/src")

import tvd.detect
print("tvd imported from:", tvd.detect.__file__)

In [ ]:
# 2. Read the Roboflow API key from Kaggle secrets
import os
from kaggle_secrets import UserSecretsClient

os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
print("ROBOFLOW_API_KEY loaded from secrets")

In [ ]:
# 3. Download the helmet dataset (helmet-detection_yolov8 v3, YOLOv8 format)
!python scripts/download_dataset.py \
    --workspace learning-evidence --project helmet-detection_yolov8 --version 3 \
    --format yolov8 --location data/helmet

In [ ]:
# 4. Download the plate dataset (License Plate Recognition v13, YOLOv8 format)
!python scripts/download_dataset.py \
    --workspace roboflow-universe-projects --project license-plate-recognition-rxg4e --version 13 \
    --format yolov8 --location data/plates

In [ ]:
# 5. Sanity-check both datasets
from tvd.detect.data import validate_dataset

for yaml_path in ("data/helmet/data.yaml", "data/plates/data.yaml"):
    i = validate_dataset(yaml_path)
    print(yaml_path, "->", i.n_train, "train /", i.n_val, "val /", i.names)

In [ ]:
# 6. Train the helmet model (20 epochs, CUDA) -> weights/helmet.pt
!python scripts/train_detector.py --train-config configs/train.helmet.yaml --device cuda

In [ ]:
# 7. Train the plate model (20 epochs, CUDA) -> weights/plate.pt
# tip: if the session time limit is close, append --epochs 10
!python scripts/train_detector.py --train-config configs/train.plate.yaml --device cuda

In [ ]:
# 8. Evaluate the helmet model -> results/detection/helmet/mAP.md + metrics.json
!python scripts/eval_detector.py --weights weights/helmet.pt \
    --train-config configs/train.helmet.yaml --device cuda

In [ ]:
# 9. Evaluate the plate model -> results/detection/plate/mAP.md + metrics.json
!python scripts/eval_detector.py --weights weights/plate.pt \
    --train-config configs/train.plate.yaml --device cuda

In [ ]:
# 10. Bundle the artifacts; download tvd_artifacts.zip from the notebook's Output tab
# (docs/RESULTS.md is overwritten by the last eval, so the per-run mAP.md files
# are the ones to keep — commit them under docs/ back on your machine.)
# no -j flag: it flattens paths and the two mAP.md files would collide
!cd /kaggle/working/tvd && zip /kaggle/working/tvd_artifacts.zip \
    weights/helmet.pt weights/plate.pt \
    results/detection/helmet/mAP.md results/detection/plate/mAP.md \
    results/detection/helmet/metrics.json results/detection/plate/metrics.json
print("download /kaggle/working/tvd_artifacts.zip from the Output tab")